## import

In [35]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from pymongo import MongoClient
from datetime import datetime
import re
from time import sleep
import time

## connect

In [36]:
client = MongoClient('mongodb://localhost:27017/')
client.drop_database('stock')
db = client['stock']


## collection

In [37]:
collection = db['MBB']

### empty 

In [38]:
all_page=[]
stocks=[]

In [39]:
#list of stock to crawl
stock_name = ['MBB','ACB','FPT','STB']

## run webdriver

In [40]:
def crawl_data():
    body = driver.find_element(By.TAG_NAME, "body")
    for _ in range(10):  
        body.send_keys(Keys.END)
        sleep(2)
    # Lấy toàn bộ hàng trong bảng
    rows = driver.find_elements(By.CSS_SELECTOR, ".simplize-table-row.simplize-table-row-level-0")

    for row in rows:
        columns = row.find_elements(By.TAG_NAME, "td")
        _date = columns[0].text
        open_price = columns[1].text.replace(",", "")
        highest_price = columns[2].text.replace(",", "")
        lowest_price = columns[3].text.replace(",", "")
        closing_price = columns[4].text.replace(",", "")
        changed_price = columns[5].text.replace(",", "")
        if (changed_price=='-'):
            changed_price = 0
        price_change_percentage = columns[6].text.replace(",", "")
        if (price_change_percentage == '-'):
            price_change_percentage = 0
        changed_volume = columns[7].text.replace(",", "")
        data = {
            "date": _date,
            "open_price": open_price,
            "highest_price": highest_price,
            "lowest_price": lowest_price,
            "closing_price": closing_price,
            "changed_price": changed_price,
            "price_change_percentage": price_change_percentage,
            "changed_volume": changed_volume
        }
        stocks.append(data)       

In [41]:
# crawl each stock
for i in stock_name:
    url = "https://simplize.vn/co-phieu/"+ i +"/lich-su-gia"
    driver = webdriver.Chrome()
    driver.get(url)
    sleep(10)
    # crawl_data()
    #crawl from each page 
    a = driver.find_element(By.XPATH , '//*[@id="phan-tich"]/div[2]/div/div/div[2]/div[1]/div/div[3]/ul/li[9]/div')

    # print(len(a))
    for i in range(1,4):
        print(i)
        sleep(4)
        crawl_data()
        page = a.click()
    #     # wait = WebDriverWait(driver, 20)
    #     # element = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="phan-tich"]/div[2]/div/div/div[2]/div[1]/div/div[3]/ul/li[9]/div')))
    #     # element.click()
    driver.quit()

In [42]:
print(len(stocks))

120


In [43]:
if stocks:
    try:
        collection.insert_many(stocks)
        print(f"Successfully insert {len(stocks)} stocks.")
    except Exception as e:
        print(f"Error insert stocks: {e}")

Successfully insert 120 stocks.


In [44]:
len(stocks)

120

## close drive

In [45]:
driver.quit()

## query

In [46]:
for demo in collection.find():
    print(demo)

{'_id': ObjectId('6714c34c2e1e061362be8537'), 'date': '18/10/2024', 'open_price': '26000', 'highest_price': '26200', 'lowest_price': '25700', 'closing_price': '25750', 'changed_price': '-150', 'price_change_percentage': '-0.58%', 'changed_volume': '18478000'}
{'_id': ObjectId('6714c34c2e1e061362be8538'), 'date': '17/10/2024', 'open_price': '25550', 'highest_price': '25900', 'lowest_price': '25350', 'closing_price': '25900', 'changed_price': '+400', 'price_change_percentage': '1.57%', 'changed_volume': '11681000'}
{'_id': ObjectId('6714c34c2e1e061362be8539'), 'date': '16/10/2024', 'open_price': '25650', 'highest_price': '25650', 'lowest_price': '25400', 'closing_price': '25500', 'changed_price': '-150', 'price_change_percentage': '-0.58%', 'changed_volume': '10829400'}
{'_id': ObjectId('6714c34c2e1e061362be853a'), 'date': '15/10/2024', 'open_price': '25950', 'highest_price': '26100', 'lowest_price': '25650', 'closing_price': '25650', 'changed_price': '-200', 'price_change_percentage': '

In [47]:
for demo in collection.find({'date': '17/10/2024'}):
    print(demo)

{'_id': ObjectId('6714c34c2e1e061362be8538'), 'date': '17/10/2024', 'open_price': '25550', 'highest_price': '25900', 'lowest_price': '25350', 'closing_price': '25900', 'changed_price': '+400', 'price_change_percentage': '1.57%', 'changed_volume': '11681000'}
{'_id': ObjectId('6714c34c2e1e061362be8556'), 'date': '17/10/2024', 'open_price': '25900', 'highest_price': '26150', 'lowest_price': '25750', 'closing_price': '26150', 'changed_price': '+400', 'price_change_percentage': '1.55%', 'changed_volume': '9195000'}
{'_id': ObjectId('6714c34c2e1e061362be8574'), 'date': '17/10/2024', 'open_price': '136500', 'highest_price': '137100', 'lowest_price': '135500', 'closing_price': '137000', 'changed_price': '+700', 'price_change_percentage': '0.51%', 'changed_volume': '2579200'}
{'_id': ObjectId('6714c34c2e1e061362be8592'), 'date': '17/10/2024', 'open_price': '33700', 'highest_price': '34600', 'lowest_price': '33250', 'closing_price': '34600', 'changed_price': '+1000', 'price_change_percentage': 

In [48]:
for demo in collection.find().sort('highest_price',-1).limit(1):
    print(demo)

{'_id': ObjectId('6714c34c2e1e061362be8591'), 'date': '18/10/2024', 'open_price': '35100', 'highest_price': '36450', 'lowest_price': '34700', 'closing_price': '35550', 'changed_price': '+950', 'price_change_percentage': '2.75%', 'changed_volume': '32868800'}


## close DB

In [49]:
# client.close()